# ✊🤞✋ 手勢識別 - MobileNetV2 + 強化對比 Notebook
使用 Sign Language Digits（只選 0,2,5）進行灰階 → RGB 處理，訓練 MobileNetV2 模型，並匯出 TFLite 模型。

## 📥 載入並前處理資料

In [ ]:
import numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# 載入資料
X_all = np.load("X.npy")  # shape: (2062, 64, 64)
Y_all = np.load("Y.npy")  # shape: (2062, 10)

# 從 one-hot 還原標籤
labels = np.argmax(Y_all, axis=1)

# 選出 0, 2, 5 對應石頭、剪刀、布
KEEP = [0, 2, 5]
LABEL_MAP = {0: 0, 2: 1, 5: 2}
selected = np.isin(labels, KEEP)
X_sel = X_all[selected]
y_sel = labels[selected]

# 灰階擴通道 → resize → normalize → RGB
X_sel = np.expand_dims(X_sel, -1).astype("float32")
X_resized = tf.image.resize(X_sel, [96, 96])
X_resized = X_resized / 255.0
X_input = tf.image.grayscale_to_rgb(X_resized)

# 處理標籤
y_mapped = np.array([LABEL_MAP[v] for v in y_sel])
y_onehot = to_categorical(y_mapped, num_classes=3)

# 分割訓練/驗證
X_train, X_test, y_train, y_test = train_test_split(
    X_input.numpy(), y_onehot, test_size=0.1, stratify=y_mapped
)


## 🧠 建立並訓練 MobileNetV2 模型

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# 資料增強
data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

# MobileNetV2 基底
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = True

# 加入分類器
model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    data_aug,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

# 開始訓練
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)


## 💾 匯出模型為 TFLite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("rps_mobilenetv2.tflite", "wb") as f:
    f.write(tflite_model)
print("✅ 已成功匯出為 rps_mobilenetv2.tflite")


## 🖼️ 預測測試集並顯示影像（強化對比）

In [ ]:
import matplotlib.pyplot as plt
preds = model.predict(X_test)

for i in range(10):
    img = X_test[i]
    normed = img / img.max()  # 強化對比再顯示
    print(f"Pixel range of image {i}: min={img.min()}, max={img.max()}")
    plt.imshow(normed)
    plt.title(f"✅ Label: {np.argmax(y_test[i])} | 🔮 Pred: {np.argmax(preds[i])}")
    plt.axis('off')
    plt.show()
